# 3. Inference: the Quetico messy-background test set

`evaluate` runs a model on particle images and writes `predictions.csv`. With labelled data it also
grades the predictions against the manual classes and draws the summary tables and charts.

This notebook runs the evaluation used for the released `noise` model on the **Quetico
messy-background test set**, and gets the same results.

| Section | Input | Model |
|---|---|---|
| A | Quetico messy-background crops, with manual classes | released `noise` model |
| B | the same crops | your model from `train.ipynb` |
| C | the same runs as a raw FlowCam export (mosaics + CSVs), no classes | released `noise` model |
| D | single images, from Python | released `noise` model |

The released weights download automatically on first use (115 MB per model) into
`~/.cache/hierroute`.

Requires `pip install hierroute jupyter`. The example data lives at `/data/Hier-Zooplankton` on the lab server; change `DATA_DIR` below if your copy is elsewhere. Everything this notebook writes goes under `WORK_DIR`.

In [ ]:
from pathlib import Path

DATA_DIR = Path("/data/Hier-Zooplankton")        # example dataset (internal)
WORK_DIR = Path.home() / "hierroute_example"      # everything the example notebooks write goes here
WORK_DIR.mkdir(parents=True, exist_ok=True)

QUETICO = DATA_DIR / "quetico" / "Messy"                                    # the test set
RUN_DIR = WORK_DIR / "training_result" / "swin_t_cnn_noise_example"         # from train.ipynb
RESULTS_DIR = WORK_DIR / "results"

In [ ]:
import pandas as pd
from IPython.display import Image as ShowImage, display
from hierroute import EvaluateOptions, evaluate
from hierroute.flowcam import ClassRules

## A. The released `noise` model on Quetico messy backgrounds

`quetico/Messy/` holds 25,564 particles from 11 Quetico lake runs, all with busy backgrounds (debris,
fibres, bubbles). They are sorted into one folder per manual class. The raw export they were cut from is
alongside, in `Classified_data/` (mosaics) and `CSV_Data/`. None of these runs were used for training.

With `labelled=True`, each folder name is the manual class, matched to the taxonomy by the hierarchy's
own rules:
- leaf names match directly
- catch-alls such as `Detritus`, `Bubble`, `Small_Nauplius` and `Decomposing` → `Zoop-No`
- coarse IDs such as `Rotifer`, `Copepod`, `Cladocera` and `Zooplankton` → that branch's `_Unknown` leaf

Quetico needs two more rules, passed as `folder_map`. Folders that match nothing (`Unknown`,
`MultiClass`, `Partial`, …) still get predictions but aren't graded.

In [ ]:
QUETICO_FOLDERS = {"Duplicate": "Zoop-No", "Kellicottia_Smalls": "Kellicottia"}

from hierroute.config import load_hierarchy
mapping = load_hierarchy("mnr_17leaf").folder_to_node(QUETICO_FOLDERS)
folders = sorted(p for p in QUETICO.iterdir() if p.is_dir() and p.name not in {"Classified_data", "CSV_Data"})
pd.DataFrame({"images": [len(list(f.glob("*.tif"))) for f in folders],
              "graded as": [mapping.get(f.name, "(not graded)") for f in folders]},
             index=[f.name for f in folders])

Besides the five default outputs, this also turns on the optional extras used for the original
evaluation:
- `zoop_only`: leaf accuracy without `Zoop-No`
- `by_sample`: one outcome bar per run
- `breakdown`: what the model calls the ungraded folders

Each crop's FlowCam measurements (the `csv_*` columns) are looked up in `CSV_Data/` by mosaic and
Particle ID. The `Cladocera/` folder also holds particles the CSVs call `Cladoceran`, so `class_rules`
tells the lookup that the two names are the same class.

In [ ]:
released = evaluate(EvaluateOptions(
    input=QUETICO,
    out_dir=RESULTS_DIR / "quetico_messy_noise",
    labelled=True,
    model="noise",
    folder_map=QUETICO_FOLDERS,
    class_rules=ClassRules(rename={"Cladoceran": "Cladocera"}),
    zoop_only=True,
    by_sample=True,
    breakdown=("Unknown", "Conochilus_Maybe", "Small_Nauplius", "Decomposing", "Decomposing_Copepod",
               "Decomposing_Calanoid", "Decomposing_Cyclopoid", "MultiClass", "Kellicottia_Multiple",
               "Partial"),
))

In [ ]:
for name in ["hierarchy_level_metrics.png", "leaf_accuracy.png", "id_outcomes_table.png",
             "id_outcomes_stacked_bar.png", "id_outcomes_stacked_bar_by_sample.png",
             "top_predictions_breakdown.png"]:
    display(ShowImage(filename=RESULTS_DIR / "quetico_messy_noise" / name, width=750))

**Expected results.** These match the original evaluation exactly:
- 23,577 of the 25,564 particles are graded
- level accuracy is 0.903 / 0.782 / 0.728 (`hierarchy_level_metrics.png`)
- leaf accuracy is 0.848 (`leaf_accuracy.png`)
- 82.2% of graded particles are `correct_all`

The ID outcome shares:

In [ ]:
OUTCOMES = ["correct_all", "correct_incomplete", "incorrect_l3", "incorrect_l2", "incorrect_all"]

def outcome_shares(df):
    graded = df[df["in_taxonomy"]]
    return graded[OUTCOMES].astype(bool).mean()

print(f"graded {int(released['in_taxonomy'].sum())} of {len(released)}")
outcome_shares(released).round(4)

`predictions.csv` has one row per image:
- `predicted_class_1`, and the runner-up `predicted_class_2`
- the prediction and the model's confidence at each level of the tree
- the ID outcome (`correct_all` … `incorrect_all`)
- the FlowCam measurements (`csv_*`)

In [ ]:
released[["image_file", "actual_class", "predicted_class_1", "predicted_class_2",
          "confidence_level_1", "confidence_level_2", "confidence_level_3",
          "predicted_level_1", "predicted_level_2", "predicted_level_3", "correct_all"]].head()

In [ ]:
# Where do the zooplankton end up? (rows: manual class as graded, columns: prediction, row %)
graded_as = released["actual_class"].map(mapping)
zoop = released[graded_as.notna() & (graded_as != "Zoop-No")]
pd.crosstab(graded_as[zoop.index], zoop["predicted_class_1"], normalize="index").round(2)

In [ ]:
# Low-confidence species calls are good candidates for manual review
released[released["confidence_level_3"] < 0.6].sort_values("confidence_level_3")[
    ["image_file", "actual_class", "predicted_class_1", "predicted_class_2", "confidence_level_3"]].head(10)

## B. Your trained model on the same test set

Pass the run folder from `train.ipynb` as `weights`; its `model_config.json` tells hierroute how to
rebuild the model. A full run of `train.ipynb` should land close to the released model, but not exactly
on it (see the note at the top of `train.ipynb`).

In [ ]:
QUETICO_OPTIONS = dict(input=QUETICO, labelled=True, folder_map=QUETICO_FOLDERS,
                       class_rules=ClassRules(rename={"Cladoceran": "Cladocera"}))

if RUN_DIR.exists():
    own = evaluate(EvaluateOptions(out_dir=RESULTS_DIR / "quetico_messy_own_model", weights=RUN_DIR,
                                   **QUETICO_OPTIONS))
    display(ShowImage(filename=RESULTS_DIR / "quetico_messy_own_model" / "hierarchy_level_metrics.png",
                      width=750))
else:
    own = None
    print(f"No trained model at {RUN_DIR} -- run train.ipynb first.")

In [ ]:
# Share of graded particles in each ID outcome, released vs your model
compare = {"released noise model": outcome_shares(released)}
if own is not None:
    compare["your model (train.ipynb)"] = outcome_shares(own)
pd.DataFrame(compare).round(4)

## C. Classify a raw FlowCam export

New data usually arrives as a raw export (mosaics plus CSVs), not as sorted crops. With
`input_type="flowcam"`, the model reads the mosaics and CSVs directly and crops each particle in memory;
there is no need to run `preprocess.ipynb` first.

Here we use the raw export the Quetico crops came from and leave `labelled` off, as you would for a new
sample. The model finds the same 25,564 particles and gives them the same classes as in section A. The
result is one row per particle with its `run`, so counts per sample are a single `crosstab`.

In [ ]:
raw = evaluate(EvaluateOptions(
    input=QUETICO / "Classified_data",      # mosaics, one folder per run
    csv_dir=QUETICO / "CSV_Data",           # the export CSVs
    input_type="flowcam",
    out_dir=RESULTS_DIR / "quetico_messy_raw",
))

In [ ]:
# Particles per predicted class in each run
counts = pd.crosstab(raw["predicted_class_1"], raw["run"])
counts.loc[counts.sum(axis=1).sort_values(ascending=False).index]

In [ ]:
# Same per-class totals as the labelled crops in section A
pd.DataFrame({"raw export": raw["predicted_class_1"].value_counts(),
              "crops (section A)": released["predicted_class_1"].value_counts()})

In [ ]:
# Optional: only trust species-level calls above a confidence threshold; otherwise report the group
confident = raw["confidence_level_3"].fillna(1) >= 0.9
raw["reported_class"] = raw["predicted_class_1"].where(
    confident, raw["predicted_level_2"].fillna(raw["predicted_level_1"]))
raw["reported_class"].value_counts()

## D. From Python, image by image

`load_model` returns the network and its config. `Predictor` gives the greedy path through the
taxonomy, the confidence at each step and a runner-up. This is useful for building your own tools on
top of hierroute.

In [ ]:
import torch
from PIL import Image
from hierroute import load_model
from hierroute.inference import Predictor, eval_transform

device = "cuda" if torch.cuda.is_available() else "cpu"
model, config = load_model("noise", device=device)
predictor = Predictor(model, device)
transform = eval_transform(config.resolution)

files = [sorted((QUETICO / c).glob("*.tif"))[0] for c in ["Calanoid", "Bosmina", "Kellicottia", "Detritus"]]
batch = torch.stack([transform(Image.open(f).convert("L")).repeat(3, 1, 1) for f in files])

for f, pred in zip(files, predictor(batch)):
    path = " > ".join(predictor.id_to_name[n] for n in pred.path[1:])
    confs = ", ".join(f"{c:.3f}" for c in pred.confidences)
    print(f"{f.parent.name:12s} -> {path:40s} confidences {confs} | runner-up {pred.predicted_2}")

## The same from the terminal

The extra folder rules and the class rename go in two small YAML files:

```bash
printf 'Duplicate: Zoop-No\nKellicottia_Smalls: Kellicottia\n' > quetico_folders.yaml
printf 'Cladoceran: Cladocera\n' > quetico_classes.yaml

# A: labelled crops, released model
hierroute evaluate /data/Hier-Zooplankton/quetico/Messy --labelled --model noise \
    --folder-map quetico_folders.yaml --class-map quetico_classes.yaml \
    --zoop-only --by-sample \
    --breakdown Unknown Conochilus_Maybe Small_Nauplius Decomposing Decomposing_Copepod \
                Decomposing_Calanoid Decomposing_Cyclopoid MultiClass Kellicottia_Multiple Partial \
    --out results/quetico_messy_noise

# B: your own model
hierroute evaluate /data/Hier-Zooplankton/quetico/Messy --labelled \
    --weights ~/hierroute_example/training_result/swin_t_cnn_noise_example \
    --folder-map quetico_folders.yaml --class-map quetico_classes.yaml --out results/quetico_messy_own_model

# C: raw export
hierroute evaluate /data/Hier-Zooplankton/quetico/Messy/Classified_data --input-type flowcam \
    --csv-dir /data/Hier-Zooplankton/quetico/Messy/CSV_Data --out results/quetico_messy_raw
```

Other optional outputs (confidence thresholds, capped tables, …) are listed under
`hierroute evaluate -h` and in the README.